In [ ]:
# ============================================================
# 07_Multi_Agent_Training.ipynb
# Specialized Agents + Attention Coordinator + MACA
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

ROOT = Path("..")
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

# ====================== CONFIG ======================
ACTION_DIM = 3          # 0: Reject, 1: Approve, 2: Counteroffer
AGENT_NAMES = ['IncomeAgent', 'BehaviorAgent', 'MacroAgent', 'ComplianceAgent']
LR = 3e-4
EPOCHS = 25
BATCH_SIZE = 64

# ====================== 1. SPECIALIZED AGENT ======================
class SpecializedAgent(nn.Module):
    def __init__(self, input_dim, action_dim=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )
    def forward(self, x):
        return self.net(x)

# ====================== 2. COORDINATOR (Attention) ======================
class CoordinatorAgent(nn.Module):
    def __init__(self, num_agents, action_dim=3):
        super().__init__()
        self.attention = nn.MultiheadAttention(
            embed_dim=action_dim, num_heads=1, batch_first=True
        )
        self.decision_net = nn.Sequential(
            nn.Linear(action_dim, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )
    def forward(self, agent_q_values):
        # agent_q_values: [num_agents, batch, action_dim]
        x = agent_q_values.permute(1, 0, 2)          # [batch, num_agents, action_dim]
        attn_out, attn_weights = self.attention(x, x, x)
        pooled = attn_out.mean(dim=1)
        final_q = self.decision_net(pooled)
        return final_q, attn_weights

# ====================== 3. LOAD DATA ======================
X = np.load(DATA_PROCESSED / "X_fused.npy")
y = np.load(DATA_PROCESSED / "y.npy")          # 0 = good, 1 = bad (adjust if needed)
thin = np.load(DATA_PROCESSED / "thin.npy")

# Convert to RL reward: +1 good repayment, -1 default
team_reward = np.where(y == 0, 1.0, -1.0).astype(np.float32)

X_train, X_test, r_train, r_test, thin_train, thin_test = train_test_split(
    X, team_reward, thin, test_size=0.25, random_state=42, stratify=y
)

STATE_DIM = X.shape[1]
print(f"Feature dimension: {STATE_DIM}")

# Simple equal feature slices (you can make them smarter later)
n = STATE_DIM // 4
feature_slices = {
    'IncomeAgent':      list(range(0, n)),
    'BehaviorAgent':    list(range(n, 2*n)),
    'MacroAgent':       list(range(2*n, 3*n)),
    'ComplianceAgent':  list(range(3*n, STATE_DIM))
}

# ====================== 4. INIT AGENTS ======================
agents = nn.ModuleDict({
    name: SpecializedAgent(len(feature_slices[name]), ACTION_DIM).to(device)
    for name in AGENT_NAMES
})
coordinator = CoordinatorAgent(len(AGENT_NAMES), ACTION_DIM).to(device)

optimizers = {name: optim.Adam(agent.parameters(), lr=LR) for name, agent in agents.items()}
coord_optimizer = optim.Adam(coordinator.parameters(), lr=LR)

# ====================== 5. MACA ======================
def compute_maca_advantages(team_rewards, attn_weights):
    baseline = team_rewards.mean()
    individual_adv = team_rewards - baseline
    ind_exp = individual_adv.unsqueeze(0).expand(len(AGENT_NAMES), -1)
    joint_adv = torch.einsum('bij,aj->ai', attn_weights.mean(0), ind_exp)
    return 0.5 * ind_exp + 0.5 * joint_adv

# ====================== 6. TRAINING ======================
train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                  torch.tensor(r_train, dtype=torch.float32)),
    batch_size=BATCH_SIZE, shuffle=True
)

print("Starting training...")
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for states, rewards in train_loader:
        states, rewards = states.to(device), rewards.to(device)

        # Forward specialized agents
        agent_qs = torch.stack([
            agents[name](states[:, feature_slices[name]]) for name in AGENT_NAMES
        ])  # [4, B, 3]

        final_q, attn_weights = coordinator(agent_qs)
        actions = final_q.argmax(dim=-1)

        advantages = compute_maca_advantages(rewards, attn_weights)

        # Update agents
        for i, name in enumerate(AGENT_NAMES):
            optimizers[name].zero_grad()
            log_probs = F.log_softmax(agent_qs[i], dim=-1)
            chosen = log_probs[range(len(actions)), actions]
            loss = -(advantages[i] * chosen).mean()
            loss.backward(retain_graph=True)
            optimizers[name].step()
            epoch_loss += loss.item()

        # Update coordinator
        coord_optimizer.zero_grad()
        target = rewards.unsqueeze(1).expand(-1, ACTION_DIM)
        coord_loss = F.mse_loss(final_q, target)
        coord_loss.backward()
        coord_optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss/len(train_loader):.4f}")

# ====================== 7. SAVE ======================
torch.save(coordinator.state_dict(), RESULTS / "multi_agent_coordinator.pt")
for name, agent in agents.items():
    torch.save(agent.state_dict(), RESULTS / f"{name}.pt")
print("✅ Models saved to results/")